# Brain Tumor Segmentation using U-Net
### Week 3 & 4 - BraTS 2021 Dataset

**Objective:** Develop a deep learning system to automatically segment brain tumors from MRI scans using U-Net architecture.

**Dataset:** BraTS 2021 - Multi-modal MRI volumes with ground-truth segmentation masks

## 1. Import Required Libraries

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from glob import glob
from tqdm import tqdm
import cv2

# TensorFlow/Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# Sklearn for train-test split
from sklearn.model_selection import train_test_split

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

TensorFlow Version: 2.20.0
GPU Available: []


## 2. Configuration and Paths

In [ ]:
# Cell: Configuration and Paths (Cell 4)

# Configuration
IMG_SIZE = 128  # Resize images to 128x128
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-4

# Dataset paths - Updated for your directory structure
DATA_PATH = os.path.join(os.getcwd(), 'BraTS2021_Training_Data')
MODEL_SAVE_PATH = os.path.join(os.getcwd(), 'models')
RESULTS_PATH = os.path.join(os.getcwd(), 'results')

# Verify data path exists
if not os.path.exists(DATA_PATH):
    print(f"WARNING: Data path does not exist: {DATA_PATH}")
    print(f"Current directory: {os.getcwd()}")
    print(f"Available directories: {os.listdir(os.getcwd())}")
else:
    print(f"Data path found: {DATA_PATH}")
    print(f"Number of patient folders: {len(os.listdir(DATA_PATH))}")

# Create directories
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)
print(f"\nModel save path: {MODEL_SAVE_PATH}")
print(f"Results path: {RESULTS_PATH}")

## 3. Load and Explore BraTS Dataset

BraTS dataset contains:
- **T1**: T1-weighted MRI
- **T1ce**: T1-weighted with contrast enhancement
- **T2**: T2-weighted MRI
- **FLAIR**: Fluid Attenuated Inversion Recovery
- **seg**: Segmentation mask (ground truth)

In [ ]:
# Cell: Load and Explore BraTS Dataset (Cell 6)

def load_nifti_file(filepath):
    """Load a NIfTI file and return the data array"""
    try:
        nifti = nib.load(filepath)
        return nifti.get_fdata()
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def explore_dataset(data_path, num_samples=3):
    """Explore the dataset structure"""
    if not os.path.exists(data_path):
        print(f"Error: Data path does not exist: {data_path}")
        return []
    
    patient_dirs = sorted([d for d in glob(os.path.join(data_path, '*')) if os.path.isdir(d)])
    print(f"Total patients: {len(patient_dirs)}")
    
    # Check first patient
    if patient_dirs:
        first_patient = patient_dirs[0]
        patient_name = os.path.basename(first_patient)
        print(f"\nFirst patient: {patient_name}")
        files = sorted(os.listdir(first_patient))
        print(f"Files in first patient directory:")
        for f in files:
            file_path = os.path.join(first_patient, f)
            file_size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
            print(f"  - {f} ({file_size:.2f} MB)")
        
        # Show a few more patients
        print(f"\nSample patient IDs:")
        for i, patient_dir in enumerate(patient_dirs[:num_samples]):
            print(f"  {i+1}. {os.path.basename(patient_dir)}")
    
    return patient_dirs

# Explore dataset - Uncomment to run
patient_dirs = explore_dataset(DATA_PATH)

In [ ]:
# New Cell: Test Data Loading

# Test loading a single patient to verify everything works
def test_single_patient(data_path, modality='flair'):
    """Test loading a single patient's data"""
    patient_dirs = sorted([d for d in glob(os.path.join(data_path, '*')) if os.path.isdir(d)])
    
    if not patient_dirs:
        print("No patient directories found!")
        return
    
    test_patient = patient_dirs[0]
    patient_name = os.path.basename(test_patient)
    print(f"Testing with patient: {patient_name}")
    print("-" * 60)
    
    try:
        # Load data
        image_vol, mask_vol = process_patient_data(test_patient, modality)
        print(f"✓ Successfully loaded patient data")
        print(f"  Image volume shape: {image_vol.shape}")
        print(f"  Mask volume shape: {mask_vol.shape}")
        print(f"  Image value range: [{image_vol.min():.3f}, {image_vol.max():.3f}]")
        print(f"  Tumor voxels: {np.sum(mask_vol > 0)} / {mask_vol.size} ({100*np.sum(mask_vol > 0)/mask_vol.size:.2f}%)")
        
        # Extract slices
        images, masks = extract_2d_slices(image_vol, mask_vol)
        print(f"✓ Extracted {len(images)} valid 2D slices")
        
        # Test resizing
        if images:
            test_img = resize_slice(images[0])
            print(f"✓ Successfully resized slice to {test_img.shape}")
        
        print("-" * 60)
        print("✓ All tests passed! Ready to process full dataset.")
        return True
        
    except Exception as e:
        print(f"✗ Error: {e}")
        import traceback
        traceback.print_exc()
        return False

# Uncomment to test
# test_single_patient(DATA_PATH, modality='flair')

## 4. Data Preprocessing - Convert 3D to 2D Slices

In [ ]:
# Cell: Data Preprocessing - Convert 3D to 2D Slices (Cell 7)

def normalize_volume(volume):
    """Normalize volume to [0, 1] range"""
    min_val = np.min(volume)
    max_val = np.max(volume)
    if max_val - min_val > 0:
        volume = (volume - min_val) / (max_val - min_val)
    return volume

def process_patient_data(patient_path, modality='flair'):
    """
    Process a single patient's data
    Args:
        patient_path: path to patient directory
        modality: which MRI modality to use ('t1', 't1ce', 't2', 'flair')
    Returns:
        image_volume: 3D MRI volume
        mask_volume: 3D segmentation mask
    """
    patient_id = os.path.basename(patient_path)
    
    # Build file paths - BraTS 2021 naming convention
    image_filename = f"{patient_id}_{modality}.nii.gz"
    mask_filename = f"{patient_id}_seg.nii.gz"
    
    image_path = os.path.join(patient_path, image_filename)
    mask_path = os.path.join(patient_path, mask_filename)
    
    # Verify files exist
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image file not found: {image_path}")
    if not os.path.exists(mask_path):
        raise FileNotFoundError(f"Mask file not found: {mask_path}")
    
    # Load MRI modality
    image_volume = load_nifti_file(image_path)
    if image_volume is None:
        raise ValueError(f"Failed to load image from {image_path}")
    
    # Load segmentation mask
    mask_volume = load_nifti_file(mask_path)
    if mask_volume is None:
        raise ValueError(f"Failed to load mask from {mask_path}")
    
    # Normalize image
    image_volume = normalize_volume(image_volume)
    
    # Convert mask to binary (tumor vs non-tumor)
    # BraTS masks have labels: 0 (background), 1 (necrotic), 2 (edema), 4 (enhancing)
    mask_volume = (mask_volume > 0).astype(np.float32)
    
    return image_volume, mask_volume

def extract_2d_slices(image_volume, mask_volume, axis=2, min_tumor_ratio=0.01):
    """
    Extract 2D slices from 3D volume along specified axis
    Args:
        image_volume: 3D MRI volume
        mask_volume: 3D mask volume
        axis: axis along which to slice (0, 1, or 2)
        min_tumor_ratio: minimum ratio of tumor pixels to include slice
    Returns:
        images: list of 2D image slices
        masks: list of 2D mask slices
    """
    images = []
    masks = []
    
    num_slices = image_volume.shape[axis]
    
    for i in range(num_slices):
        if axis == 0:
            img_slice = image_volume[i, :, :]
            mask_slice = mask_volume[i, :, :]
        elif axis == 1:
            img_slice = image_volume[:, i, :]
            mask_slice = mask_volume[:, i, :]
        else:  # axis == 2
            img_slice = image_volume[:, :, i]
            mask_slice = mask_volume[:, :, i]
        
        # Skip slices with very little or no tumor
        tumor_ratio = np.sum(mask_slice) / mask_slice.size
        if tumor_ratio >= min_tumor_ratio:
            images.append(img_slice)
            masks.append(mask_slice)
    
    return images, masks

## 5. Resize and Prepare Dataset

In [ ]:
# Cell: Resize and Prepare Dataset (Cell 8)

def resize_slice(img, size=(IMG_SIZE, IMG_SIZE)):
    """Resize a 2D slice to specified size"""
    return cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)

def prepare_full_dataset(data_path, num_patients=None, modality='flair'):
    """
    Prepare the complete dataset by processing all patients
    Args:
        data_path: path to BraTS dataset
        num_patients: number of patients to process (None = all)
        modality: MRI modality to use
    Returns:
        X: array of image slices
        y: array of mask slices
    """
    # Get patient directories
    patient_dirs = sorted([d for d in glob(os.path.join(data_path, '*')) if os.path.isdir(d)])
    
    if len(patient_dirs) == 0:
        raise ValueError(f"No patient directories found in {data_path}")
    
    if num_patients:
        patient_dirs = patient_dirs[:num_patients]
    
    all_images = []
    all_masks = []
    failed_patients = []
    
    print(f"Processing {len(patient_dirs)} patients from: {data_path}")
    print(f"Using modality: {modality}")
    print("-" * 60)
    
    for patient_path in tqdm(patient_dirs, desc="Processing patients"):
        try:
            # Load and process patient data
            image_vol, mask_vol = process_patient_data(patient_path, modality)
            
            # Extract 2D slices
            images, masks = extract_2d_slices(image_vol, mask_vol)
            
            if len(images) == 0:
                print(f"\nWarning: No valid slices found for {os.path.basename(patient_path)}")
                continue
            
            # Resize slices
            for img, msk in zip(images, masks):
                img_resized = resize_slice(img)
                msk_resized = resize_slice(msk)
                all_images.append(img_resized)
                all_masks.append(msk_resized)
        
        except Exception as e:
            patient_name = os.path.basename(patient_path)
            failed_patients.append(patient_name)
            print(f"\nError processing {patient_name}: {e}")
            continue
    
    if len(all_images) == 0:
        raise ValueError("No images were successfully processed!")
    
    # Convert to numpy arrays and add channel dimension
    X = np.array(all_images)[..., np.newaxis]  # Shape: (N, H, W, 1)
    y = np.array(all_masks)[..., np.newaxis]   # Shape: (N, H, W, 1)
    
    print("\n" + "="*60)
    print("Dataset Preparation Summary:")
    print("="*60)
    print(f"Images shape: {X.shape}")
    print(f"Masks shape: {y.shape}")
    print(f"Total slices: {len(X)}")
    print(f"Patients processed successfully: {len(patient_dirs) - len(failed_patients)}/{len(patient_dirs)}")
    if failed_patients:
        print(f"Failed patients: {', '.join(failed_patients)}")
    print(f"Average slices per patient: {len(X) / (len(patient_dirs) - len(failed_patients)):.1f}")
    print("="*60)
    
    return X, y

# Prepare dataset - Start with small subset for testing
# Uncomment the line below to run
# X, y = prepare_full_dataset(DATA_PATH, num_patients=10, modality='flair')

## 6. Visualize Sample Data

In [ ]:
def visualize_samples(X, y, num_samples=5):
    """Visualize random samples from the dataset"""
    indices = np.random.choice(len(X), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    
    for i, idx in enumerate(indices):
        # Original image
        axes[i, 0].imshow(X[idx, :, :, 0], cmap='gray')
        axes[i, 0].set_title('MRI Slice')
        axes[i, 0].axis('off')
        
        # Ground truth mask
        axes[i, 1].imshow(y[idx, :, :, 0], cmap='jet')
        axes[i, 1].set_title('Ground Truth Mask')
        axes[i, 1].axis('off')
        
        # Overlay
        axes[i, 2].imshow(X[idx, :, :, 0], cmap='gray')
        axes[i, 2].imshow(y[idx, :, :, 0], cmap='jet', alpha=0.4)
        axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize samples
# visualize_samples(X, y, num_samples=5)

## 7. Train-Test Split

In [ ]:
def split_dataset(X, y, test_size=0.2, val_size=0.1):
    """
    Split dataset into train, validation, and test sets
    """
    # First split: train+val vs test
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # Second split: train vs val
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42
    )
    
    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Split the data
# X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(X, y)

## 8. Build U-Net Architecture

In [ ]:
def conv_block(inputs, num_filters):
    """
    Convolutional block: Conv2D -> BatchNorm -> ReLU -> Conv2D -> BatchNorm -> ReLU
    """
    x = layers.Conv2D(num_filters, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(num_filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    return x

def encoder_block(inputs, num_filters):
    """
    Encoder block: Conv block -> MaxPooling
    """
    x = conv_block(inputs, num_filters)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters):
    """
    Decoder block: UpSampling -> Concatenate with skip connection -> Conv block
    """
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 1)):
    """
    Build U-Net model
    """
    inputs = layers.Input(input_shape)
    
    # Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)
    
    # Bottleneck
    b1 = conv_block(p4, 1024)
    
    # Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)
    
    # Output layer
    outputs = layers.Conv2D(1, 1, padding='same', activation='sigmoid')(d4)
    
    model = models.Model(inputs, outputs, name='U-Net')
    return model

# Build the model
model = build_unet()
model.summary()

## 9. Define Custom Metrics and Loss Functions

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Dice coefficient metric
    """
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """
    Dice loss function
    """
    return 1 - dice_coefficient(y_true, y_pred)

def iou_metric(y_true, y_pred, smooth=1e-6):
    """
    Intersection over Union (IoU) metric
    """
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def combined_loss(y_true, y_pred):
    """
    Combined loss: Binary Cross-Entropy + Dice Loss
    """
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

## 10. Compile the Model

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=combined_loss,
    metrics=[dice_coefficient, iou_metric, 'accuracy']
)

print("Model compiled successfully!")

## 11. Define Callbacks

In [ ]:
# Define callbacks
checkpoint = ModelCheckpoint(
    os.path.join(MODEL_SAVE_PATH, 'unet_best.h5'),
    monitor='val_dice_coefficient',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_dice_coefficient',
    patience=10,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [checkpoint, early_stop, reduce_lr]

## 12. Train the Model

In [ ]:
# Train the model
# history = model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     batch_size=BATCH_SIZE,
#     epochs=EPOCHS,
#     callbacks=callbacks,
#     verbose=1
# )

# Save final model
# model.save(os.path.join(MODEL_SAVE_PATH, 'unet_final.h5'))
# print("Training complete!")

## 13. Plot Training History

In [ ]:
def plot_training_history(history):
    """
    Plot training and validation metrics
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Dice Coefficient
    axes[0, 1].plot(history.history['dice_coefficient'], label='Train Dice')
    axes[0, 1].plot(history.history['val_dice_coefficient'], label='Val Dice')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # IoU
    axes[1, 0].plot(history.history['iou_metric'], label='Train IoU')
    axes[1, 0].plot(history.history['val_iou_metric'], label='Val IoU')
    axes[1, 0].set_title('IoU Metric')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('IoU')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Accuracy
    axes[1, 1].plot(history.history['accuracy'], label='Train Acc')
    axes[1, 1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1, 1].set_title('Accuracy')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, 'training_history.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Plot training history
# plot_training_history(history)

## 14. Evaluate Model on Test Set

In [ ]:
def evaluate_model(model, X_test, y_test):
    """
    Evaluate model on test set
    """
    print("Evaluating model on test set...")
    results = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=1)
    
    print("\n" + "="*50)
    print("Test Set Results:")
    print("="*50)
    print(f"Loss: {results[0]:.4f}")
    print(f"Dice Coefficient: {results[1]:.4f}")
    print(f"IoU: {results[2]:.4f}")
    print(f"Accuracy: {results[3]:.4f}")
    print("="*50)
    
    return results

# Evaluate on test set
# test_results = evaluate_model(model, X_test, y_test)

## 15. Make Predictions and Visualize Results

In [ ]:
def predict_and_visualize(model, X_test, y_test, num_samples=10, threshold=0.5):
    """
    Make predictions and visualize results
    """
    # Select random samples
    indices = np.random.choice(len(X_test), num_samples, replace=False)
    
    # Make predictions
    predictions = model.predict(X_test[indices], batch_size=BATCH_SIZE)
    predictions = (predictions > threshold).astype(np.float32)
    
    # Visualize
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    for i, idx in enumerate(indices):
        # Original image
        axes[i, 0].imshow(X_test[idx, :, :, 0], cmap='gray')
        axes[i, 0].set_title('MRI Slice')
        axes[i, 0].axis('off')
        
        # Ground truth
        axes[i, 1].imshow(y_test[idx, :, :, 0], cmap='jet')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(predictions[i, :, :, 0], cmap='jet')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(X_test[idx, :, :, 0], cmap='gray')
        axes[i, 3].imshow(predictions[i, :, :, 0], cmap='jet', alpha=0.4)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
        
        # Calculate metrics for this sample
        dice = dice_coefficient(y_test[idx:idx+1], predictions[i:i+1]).numpy()
        iou = iou_metric(y_test[idx:idx+1], predictions[i:i+1]).numpy()
        axes[i, 3].text(0.5, -0.1, f'Dice: {dice:.3f} | IoU: {iou:.3f}',
                       transform=axes[i, 3].transAxes, ha='center')
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, 'predictions.png'), dpi=300, bbox_inches='tight')
    plt.show()

# Visualize predictions
# predict_and_visualize(model, X_test, y_test, num_samples=10)

## 16. Calculate Per-Sample Metrics

In [ ]:
def calculate_per_sample_metrics(model, X_test, y_test, threshold=0.5):
    """
    Calculate Dice and IoU for each test sample
    """
    predictions = model.predict(X_test, batch_size=BATCH_SIZE)
    predictions = (predictions > threshold).astype(np.float32)
    
    dice_scores = []
    iou_scores = []
    
    for i in range(len(X_test)):
        dice = dice_coefficient(y_test[i:i+1], predictions[i:i+1]).numpy()
        iou = iou_metric(y_test[i:i+1], predictions[i:i+1]).numpy()
        dice_scores.append(dice)
        iou_scores.append(iou)
    
    dice_scores = np.array(dice_scores)
    iou_scores = np.array(iou_scores)
    
    print("\nPer-Sample Metrics:")
    print(f"Dice - Mean: {dice_scores.mean():.4f}, Std: {dice_scores.std():.4f}, Min: {dice_scores.min():.4f}, Max: {dice_scores.max():.4f}")
    print(f"IoU  - Mean: {iou_scores.mean():.4f}, Std: {iou_scores.std():.4f}, Min: {iou_scores.min():.4f}, Max: {iou_scores.max():.4f}")
    
    # Plot distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(dice_scores, bins=50, edgecolor='black')
    axes[0].set_title('Dice Score Distribution')
    axes[0].set_xlabel('Dice Score')
    axes[0].set_ylabel('Frequency')
    axes[0].axvline(dice_scores.mean(), color='red', linestyle='--', label=f'Mean: {dice_scores.mean():.3f}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(iou_scores, bins=50, edgecolor='black')
    axes[1].set_title('IoU Score Distribution')
    axes[1].set_xlabel('IoU Score')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(iou_scores.mean(), color='red', linestyle='--', label=f'Mean: {iou_scores.mean():.3f}')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PATH, 'metrics_distribution.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    return dice_scores, iou_scores

# Calculate per-sample metrics
# dice_scores, iou_scores = calculate_per_sample_metrics(model, X_test, y_test)

## 17. Save Results Summary

In [ ]:
def save_results_summary(test_results, dice_scores, iou_scores):
    """
    Save a summary of results to a text file
    """
    with open(os.path.join(RESULTS_PATH, 'results_summary.txt'), 'w') as f:
        f.write("="*60 + "\n")
        f.write("Brain Tumor Segmentation - Results Summary\n")
        f.write("="*60 + "\n\n")
        
        f.write("Model: U-Net\n")
        f.write(f"Image Size: {IMG_SIZE}x{IMG_SIZE}\n")
        f.write(f"Batch Size: {BATCH_SIZE}\n")
        f.write(f"Epochs: {EPOCHS}\n")
        f.write(f"Learning Rate: {LEARNING_RATE}\n\n")
        
        f.write("Test Set Results:\n")
        f.write("-" * 60 + "\n")
        f.write(f"Loss: {test_results[0]:.4f}\n")
        f.write(f"Dice Coefficient: {test_results[1]:.4f}\n")
        f.write(f"IoU: {test_results[2]:.4f}\n")
        f.write(f"Accuracy: {test_results[3]:.4f}\n\n")
        
        f.write("Per-Sample Statistics:\n")
        f.write("-" * 60 + "\n")
        f.write(f"Dice - Mean: {dice_scores.mean():.4f}, Std: {dice_scores.std():.4f}\n")
        f.write(f"IoU  - Mean: {iou_scores.mean():.4f}, Std: {iou_scores.std():.4f}\n")
        f.write("="*60 + "\n")
    
    print(f"Results summary saved to {RESULTS_PATH}results_summary.txt")

# Save results
# save_results_summary(test_results, dice_scores, iou_scores)

## 18. Load and Test Saved Model (Optional)

In [ ]:
# Load saved model
# loaded_model = keras.models.load_model(
#     os.path.join(MODEL_SAVE_PATH, 'unet_best.h5'),
#     custom_objects={
#         'combined_loss': combined_loss,
#         'dice_coefficient': dice_coefficient,
#         'iou_metric': iou_metric
#     }
# )
# print("Model loaded successfully!")

## Conclusion

This notebook demonstrates a complete pipeline for brain tumor segmentation:

1. **Data Loading**: Load BraTS dataset with multi-modal MRI scans
2. **Preprocessing**: Convert 3D volumes to 2D slices, normalize, and resize
3. **Model Architecture**: Implement U-Net with encoder-decoder structure
4. **Training**: Train with combined loss (BCE + Dice)
5. **Evaluation**: Evaluate using Dice Score and IoU metrics
6. **Visualization**: Compare predictions with ground truth

### Next Steps:
- Experiment with data augmentation
- Try different architectures (Attention U-Net, U-Net++)
- Use multi-modal inputs (combine T1, T2, FLAIR)
- Implement 3D U-Net for volumetric segmentation
- Fine-tune hyperparameters